# RMSE Optimized Traffic Demand Prediction
CatBoost + LightGBM + XGBoost + Ridge Stacking


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Load datasets
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# Log transform target for RMSE stability
y = np.log1p(train['demand'])

def feature_engineering(df, train_ref):
    df = df.copy()
    stats = train_ref.groupby('geohash')['demand'].agg(['mean','std','max']).fillna(0)
    stats.columns=['geo_mean','geo_std','geo_max']
    km = KMeans(n_clusters=15, random_state=42, n_init=10)
    stats['cluster']=km.fit_predict(StandardScaler().fit_transform(stats))
    df = df.merge(stats,on='geohash',how='left')
    df['hour']=df['timestamp'].str.split(':').str[0].astype(int)
    df['minute_total']=df['timestamp'].str.split(':').str[0].astype(int)*60 + df['timestamp'].str.split(':').str[1].astype(int)
    df['sin24']=np.sin(2*np.pi*df['minute_total']/(24*60))
    df['cos24']=np.cos(2*np.pi*df['minute_total']/(24*60))
    for c in ['RoadType','LargeVehicles','Landmarks','Weather','cluster']:
        df[c]=df[c].astype(str).astype('category')
    return df

X_train = feature_engineering(train.drop(columns=['demand']), train)
X_test = feature_engineering(test, train)

features=[c for c in X_train.columns if c not in ['Index','geohash','timestamp']]
cat_features=[c for c in features if str(X_train[c].dtype)=='category']

oof=np.zeros((len(X_train),3))
preds=np.zeros((len(X_test),3))

kf=KFold(n_splits=5,shuffle=True,random_state=42)

for fold,(trn,val) in enumerate(kf.split(X_train)):
    X_tr=X_train[features].iloc[trn]
    X_val=X_train[features].iloc[val]
    y_tr=y.iloc[trn]

    lgbm=lgb.LGBMRegressor(n_estimators=2000,learning_rate=0.03)
    lgbm.fit(X_tr,y_tr)
    oof[val,0]=lgbm.predict(X_val)
    preds[:,0]+=lgbm.predict(X_test[features])/5

    cat=CatBoostRegressor(iterations=2000,learning_rate=0.03,depth=10,verbose=0,cat_features=cat_features)
    cat.fit(X_tr,y_tr)
    oof[val,1]=cat.predict(X_val)
    preds[:,1]+=cat.predict(X_test[features])/5

    Xtr2=X_tr.copy(); Xval2=X_val.copy(); Xte2=X_test[features].copy()
    for c in cat_features:
        Xtr2[c]=Xtr2[c].cat.codes
        Xval2[c]=Xval2[c].cat.codes
        Xte2[c]=Xte2[c].cat.codes

    xgbm=xgb.XGBRegressor(n_estimators=2000,max_depth=10,learning_rate=0.03)
    xgbm.fit(Xtr2,y_tr)
    oof[val,2]=xgbm.predict(Xval2)
    preds[:,2]+=xgbm.predict(Xte2)/5

meta=RidgeCV(alphas=[0.1,1,10])
meta.fit(oof,y)
final=np.expm1(meta.predict(preds))

submission=pd.DataFrame({'Index':test['Index'],'demand':np.maximum(0,final)})
submission.to_csv('submission.csv',index=False)
print('submission.csv generated')
